> **Reused upstream sample; local workshop adaptation.** Source: [Azure-Samples/azure-ai-content-understanding-python](https://github.com/Azure-Samples/azure-ai-content-understanding-python/blob/19f1a9d68ef5de66abef056564f32e4bc49f4067/notebooks/classifier.ipynb).
> Original commit: `19f1a9d68ef5de66abef056564f32e4bc49f4067`. MIT notice: [LICENSE](../LICENSE).
> Local changes: repo .venv/.env and Entra-only auth; read-only defaults; existing configured completion model; compact display previews with complete JSON saved by upstream helpers; explicit opt-in sample cleanup. Analysis calls incur Azure usage. No new processing framework.


# Azure AI Content Understanding - Classifier and Analyzer Demo

This notebook demonstrates how to use the Azure AI Content Understanding service to:
1. Create a classifier for document categorization
2. Create a custom analyzer to extract specific fields
3. Combine the classifier and analyzers to classify, optionally split, and analyze documents within a flexible processing pipeline

For more detailed information before getting started, please refer to the official documentation:
[Understanding Classifiers in Azure AI Services](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/classifier)

## Prerequisites
1. Ensure the Azure AI service is configured by following the [setup steps](../README.md#configure-azure-ai-service-resource).
2. Install the required packages to run this sample.

Use the repository's existing **.venv**. This upstream install cell is intentionally not executed; install only a specifically missing package in a visible terminal.


## Use the existing upstream REST client
These educational demos retain the upstream REST wrapper and analysis/schema code. This local setup reads the root .env, uses the shared .venv and Microsoft Entra ID only. The claims app continues to use the official SDK.


In [15]:
from datetime import datetime
from pathlib import Path
import logging
import json
import os
import sys
import time
import asyncio
from typing import Any, Optional
from dotenv import find_dotenv, load_dotenv

ENV_FILE = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(ENV_FILE, override=True)
NOTEBOOK_DIR = ENV_FILE.parent / "demo" / "content-understanding" / "notebooks"
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR.parent / "python"))
from content_understanding_client import AzureContentUnderstandingClient
from extension.sample_helper import save_json_to_file
from azure.identity import DefaultAzureCredential

if Path(sys.executable).resolve() != (ENV_FILE.parent / ".venv" / "Scripts" / "python.exe").resolve():
    raise RuntimeError("Select the repository .venv kernel.")
logging.basicConfig(level=logging.WARNING)
AZURE_AI_ENDPOINT = os.environ["APP_CONTENT_UNDERSTANDING_ENDPOINT"]
API_VERSION = os.getenv("CU_API_VERSION", "2025-11-01")
credential = DefaultAzureCredential()
def token_provider():
    return credential.get_token("https://cognitiveservices.azure.com/.default").token

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT, api_version=API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python-sample-ga",
)
RUN_SAMPLE_CLEANUP = False
print("Official upstream REST sample |", API_VERSION, "| Microsoft Entra ID")
print("Shared repo environment:", ENV_FILE.name, "| Kernel:", Path(sys.executable).relative_to(ENV_FILE.parent))


Official upstream REST sample | 2025-11-01 | Microsoft Entra ID
Shared repo environment: .env | Kernel: .venv\Scripts\python.exe


## Inspect existing model configuration (read-only)
The upstream default-update example is replaced here with a read-only check. No models are deployed and no shared defaults are changed. Custom examples use a model already mapped to the configured completion deployment.


In [16]:
defaults = client.get_defaults()
model_deployments = defaults.get("modelDeployments", {})
print("Existing resource model mappings (read-only):", model_deployments)
completion_deployment = os.environ["APP_CU_COMPLETION_DEPLOYMENT"]
completion_models = [model for model, deployment in model_deployments.items() if deployment == completion_deployment]
if not completion_models:
    raise ValueError("The configured completion deployment is not mapped in this account. Review resource defaults; this notebook will not patch them.")
COMPLETION_MODEL = completion_deployment if completion_deployment in completion_models else completion_models[0]
print("Custom sample completion model:", COMPLETION_MODEL, "| Existing deployment:", completion_deployment)


Existing resource model mappings (read-only): {'gpt-5.2': 'gpt-5.2', 'text-embedding-3-large': 'text-embedding-3-large', 'prebuilt-analyzer-completion': 'gpt-5.2', 'prebuilt-analyzer-completion-mini': 'gpt-5.2', 'prebuilt-analyzer-embedding': 'text-embedding-3-large'}
Custom sample completion model: gpt-5.2 | Existing deployment: gpt-5.2


## Create a Basic Classifier
Classify document from URL using begin_classify API.

High-level steps:
1. Create a custom classifier
2. Classify a document from a remote URL
3. Save the classification result to a file
4. Clean up the created classifier

In Azure AI Content Understanding, classification is integrated directly into the analyzer operation rather than requiring a separate API. To create a classifier, you define **`contentCategories`** within the analyzer's configuration, specifying up to 200 category names and descriptions that the service will use to categorize your input files. 

The **`enableSegment`** parameter controls how the classifier handles multi-document files: when set to `true`, it automatically splits and classifies different document types within a single file (useful for processing combined documents like a loan application package containing multiple forms), while setting it to `false` treats the entire file as a single document. 

For more detailed information about classification capabilities, best practices, and advanced scenarios, see the [Content Understanding classification documentation](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/concepts/classifier).

In [17]:
# Generate a unique classifier ID
analyzer_id = f"notebooks_sample_classifier_{int(time.time())}"

# Define the classifier as a dictionary
content_analyzer = {
    "baseAnalyzerId": "prebuilt-document",
    "description": f"Custom classifier for URL classification demo: {analyzer_id}",
    "config": {
        "returnDetails": True,
        "enableSegment": True,
        "contentCategories": {
            "Loan application": {
                "description": "Documents submitted by individuals or businesses to request funding, typically including personal or business details, financial history, loan amount, purpose, and supporting documentation."
            },
            "Invoice": {
                "description": "Billing documents issued by sellers or service providers to request payment for goods or services, detailing items, prices, taxes, totals, and payment terms."
            },
            "Bank_Statement": {
                "description": "Official statements issued by banks that summarize account activity over a period, including deposits, withdrawals, fees, and balances."
            }
        }
    },
    "models": {"completion": COMPLETION_MODEL},
    "tags": {"demo_type": "url_classification"}
}

# Create a custom classifier
print(f"Creating custom classifier '{analyzer_id}'...")

# Start the classifier creation operation
response = client.begin_create_analyzer(
    analyzer_id=analyzer_id,
    analyzer_template=content_analyzer,
)

# Wait for the classifier to be created
print(f"Waiting for classifier creation to complete...")
client.poll_result(response)
print(f"Classifier '{analyzer_id}' created successfully!")


Creating custom classifier 'notebooks_sample_classifier_1793954216'...
Waiting for classifier creation to complete...
Classifier 'notebooks_sample_classifier_1793954216' created successfully!


## Classify Your Document

Now, use the classifier to categorize your document.

In [18]:
# Read the mixed financial docs PDF file
pdf_path = "../data/mixed_financial_docs.pdf"
print(f"Reading document file: {pdf_path}")

# Begin binary classification operation
print(f"Starting binary classification with classifier '{analyzer_id}'...")
analysis_response = client.begin_analyze_binary(
    analyzer_id=analyzer_id,
    file_location=pdf_path,
)

# Wait for analysis completion
print(f"Waiting for document analysis to complete...")
analysis_result = client.poll_result(analysis_response)
print(f"Document analysis completed successfully!")


Reading document file: ../data/mixed_financial_docs.pdf
Starting binary classification with classifier 'notebooks_sample_classifier_1793954216'...
Waiting for document analysis to complete...
Document analysis completed successfully!


## View Classification Results

Review the classification results generated for your document.

In [19]:
# Display results
if analysis_result and "result" in analysis_result:
    result = analysis_result["result"]
    contents = result.get("contents", [])
    
    if contents:
        first_content = contents[0]
        
        # Display classification results from segments
        segments = first_content.get("segments", [])
        if segments:
            print("\nClassification Results:")
            print("=" * 50)
            for idx, segment in enumerate(segments, 1):
                print(f"\nSegment {idx}:")
                print(f"   Category: {segment.get('category', 'N/A')}")
                print(f"   Start Page: {segment.get('startPageNumber', 'N/A')}")
                print(f"   End Page: {segment.get('endPageNumber', 'N/A')}")
                print(f"   Segment ID: {segment.get('segmentId', 'N/A')}")
            print("=" * 50)
    else:
        print("No contents available in analysis result")
else:
    print("No analysis result available")



Classification Results:

Segment 1:
   Category: Invoice
   Start Page: 1
   End Page: 1
   Segment ID: segment1

Segment 2:
   Category: Bank_Statement
   Start Page: 2
   End Page: 3
   Segment ID: segment2

Segment 3:
   Category: Loan application
   Start Page: 4
   End Page: 4
   Segment ID: segment3


## Saving Classification Results
The classification result is saved to a JSON file for later analysis.

In [20]:
# Save the analysis result to a file
saved_file_path = save_json_to_file(analysis_result, filename_prefix="classification_get_result")
# Print the full analysis result as a JSON string
print(json.dumps(analysis_result, indent=2)[:1800])


Analysis result saved to: test_output\classification_get_result_20261106_083701.json
{
  "id": "83a97dac-e755-4e57-8671-509b5fb6f18f",
  "status": "Succeeded",
  "result": {
    "analyzerId": "notebooks_sample_classifier_1793954216",
    "apiVersion": "2025-11-01",
    "createdAt": "2026-11-06T08:36:58Z",
    "stringEncoding": "codePoint",
    "warnings": [],
    "contents": [
      {
        "path": "input1",
        "markdown": "CONTOSO LTD.\n\n\n# INVOICE\n\nContoso Headquarters\n123 456th St\nNew York, NY, 10001\n\nINVOICE: INV-100\n\nINVOICE DATE: 11/15/2019\n\nDUE DATE: 12/15/2019\n\nCUSTOMER NAME: MICROSOFT CORPORATION\n\nSERVICE PERIOD: 10/14/2019 - 11/14/2019\n\nCUSTOMER ID: CID-12345\n\nMicrosoft Corp\n123 Other St,\nRedmond WA, 98052\n\nBILL TO:\nMicrosoft Finance\n123 Bill St,\nRedmond WA, 98052\n\nSHIP TO:\nMicrosoft Delivery\n123 Ship St,\nRedmond WA, 98052\n\nSERVICE ADDRESS:\nMicrosoft Services\n123 Service St,\nRedmond WA, 98052\n\n\n<table>\n<tr>\n<th>SALESPERSON</th>

## Clean up the created analyzer 
After the demo completes, the classifier is automatically deleted to prevent resource accumulation.

In [21]:
if RUN_SAMPLE_CLEANUP:
    # Clean up the created classifier
    print(f"Deleting classifier '{analyzer_id}'...")
    client.delete_analyzer(analyzer_id=analyzer_id)
    print(f"Classifier '{analyzer_id}' deleted successfully!")
else:
    print('Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.')



Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.


## Create a Custom Analyzer (Advanced)

Create a custom analyzer to extract specific fields from documents.
This example extracts common fields from loan application documents and generates document excerpts.

In [22]:
# Generate a unique analyzer ID for loan applications
loan_analyzer_id = f"notebooks_sample_loan_analyzer_{int(time.time())}"

# Define custom analyzer as a dictionary
custom_analyzer = {
    "baseAnalyzerId": "prebuilt-document",
    "description": "Loan application analyzer - extracts key information from loan applications",
    "config": {
        "returnDetails": True,
        "enableLayout": True,
        "enableFormula": False,
        "estimateFieldSourceAndConfidence": True
    },
    "fieldSchema": {
        "fields": {
            "ApplicationDate": {
                "type": "date",
                "method": "generate",
                "description": "The date when the loan application was submitted."
            },
            "ApplicantName": {
                "type": "string",
                "method": "generate",
                "description": "Full name of the loan applicant or company."
            },
            "LoanAmountRequested": {
                "type": "number",
                "method": "generate",
                "description": "The total loan amount requested by the applicant."
            },
            "LoanPurpose": {
                "type": "string",
                "method": "generate",
                "description": "The stated purpose or reason for the loan."
            },
            "CreditScore": {
                "type": "number",
                "method": "generate",
                "description": "Credit score of the applicant, if available."
            },
            "Summary": {
                "type": "string",
                "method": "generate",
                "description": "A brief summary overview of the loan application details."
            }
        }
    },
    "models": {"completion": COMPLETION_MODEL},
    "tags": {"demo": "loan-application"}
}

# Create the custom analyzer
print(f"Creating custom analyzer '{loan_analyzer_id}'...")
response = client.begin_create_analyzer(
    analyzer_id=loan_analyzer_id,
    analyzer_template=custom_analyzer,
)
client.poll_result(response)
print(f"Analyzer '{loan_analyzer_id}' created successfully!")


Creating custom analyzer 'notebooks_sample_loan_analyzer_1793954221'...
Analyzer 'notebooks_sample_loan_analyzer_1793954221' created successfully!


## Create an Enhanced Classifier with Custom Analyzer

Now create a new classifier that uses the prebuilt invoice analyzer for invoices and the custom analyzer for loan application documents.
This combines document classification with field extraction in one operation.

In [23]:
# Generate a unique enhanced classifier ID
enhanced_classifier_id = f"notebooks_sample_enhanced_classifier_{int(time.time())}"

# Define enhanced classifier with custom analyzer for loan applications
enhanced_analyzer = {
    "baseAnalyzerId": "prebuilt-document",
    "description": f"Enhanced classifier with custom loan analyzer: {enhanced_classifier_id}",
    "config": {
        "returnDetails": True,
        "enableSegment": True,
        "contentCategories": {
            "Loan application": {
                "description": "Documents submitted by individuals or businesses to request funding, typically including personal or business details, financial history, loan amount, purpose, and supporting documentation.",
                "analyzerId": loan_analyzer_id  # Use the custom loan analyzer
            },
            "Invoice": {
                "description": "Billing documents issued by sellers or service providers to request payment for goods or services, detailing items, prices, taxes, totals, and payment terms."
            },
            "Bank_Statement": {
                "description": "Official statements issued by banks that summarize account activity over a period, including deposits, withdrawals, fees, and balances."
            }
        }
    },
    "models": {"completion": COMPLETION_MODEL},
    "tags": {"demo_type": "enhanced_classification"}
}

# Create the enhanced classifier
print(f"Creating enhanced classifier '{enhanced_classifier_id}'...")
response = client.begin_create_analyzer(
    analyzer_id=enhanced_classifier_id,
    analyzer_template=enhanced_analyzer,
)

# Wait for the classifier to be created
print(f"Waiting for classifier creation to complete...")
client.poll_result(response)
print(f"Enhanced classifier '{enhanced_classifier_id}' created successfully!")


Creating enhanced classifier 'notebooks_sample_enhanced_classifier_1793954225'...
Waiting for classifier creation to complete...
Enhanced classifier 'notebooks_sample_enhanced_classifier_1793954225' created successfully!


## Process Document with Enhanced Classifier

Process the document again using the enhanced classifier.
Invoices and loan applications will now have additional fields extracted.

In [10]:
pdf_path = "../data/mixed_financial_docs.pdf"
print(f"Reading document file: {pdf_path}")

# Begin binary classification operation with enhanced classifier
print(f"Starting binary classification with enhanced classifier '{enhanced_classifier_id}'...")
enhanced_analysis_response = client.begin_analyze_binary(
    analyzer_id=enhanced_classifier_id,
    file_location=pdf_path,
)

# Wait for classification completion
print(f"Waiting for classification to complete...")
enhanced_analysis_result = client.poll_result(enhanced_analysis_response)
print(f"Classification completed successfully!")


Reading document file: ../data/mixed_financial_docs.pdf
Starting binary classification with enhanced classifier 'notebooks_sample_enhanced_classifier_1793954201'...
Waiting for classification to complete...
Classification completed successfully!


## View Enhanced Results with Extracted Fields

Review the classification results alongside extracted fields from loan application documents.

In [11]:
# Display enhanced classification results
if enhanced_analysis_result and "result" in enhanced_analysis_result:
    result = enhanced_analysis_result["result"]
    contents = result.get("contents", [])
    
    if contents:
        print("\nEnhanced Classification Results with Field Extraction:")
        print("=" * 80)
        
        for idx, content_item in enumerate(contents, 1):
            print(f"\nSegment {idx}:")
            print(f"   Category: {content_item.get('category', 'N/A')}")
            print(f"   Pages: {content_item.get('startPageNumber', 'N/A')} - {content_item.get('endPageNumber', 'N/A')}")
            
            # Display extracted fields if available
            fields = content_item.get("fields", {})
            if fields:
                print(f"\n   Extracted Fields:")
                for field_name, field_value in fields.items():
                    field_type = field_value.get("type")
                    if field_type == "string":
                        print(f"      • {field_name}: {field_value.get('valueString')}")
                    elif field_type == "number":
                        print(f"      • {field_name}: {field_value.get('valueNumber')}")
                    elif field_type == "date":
                        print(f"      • {field_name}: {field_value.get('valueDate')}")
            else:
                print(f"   (No custom fields extracted for this category)")
        
        
        print("\n" + "=" * 80)
        
        # Display document information for the first segment
        first_content = contents[0]
        if first_content.get("kind") == "document":
            print(f"\nDocument Information:")
            pages = first_content.get("pages")
            if pages:
                print(f"Total pages in document: {len(pages)}")
                unit = first_content.get("unit", "units")
                print(f"Page dimensions: {pages[0].get('width')} x {pages[0].get('height')} {unit}")
    else:
        print("No contents available in enhanced analysis result")
else:
    print("No enhanced analysis result available")



Enhanced Classification Results with Field Extraction:

Segment 1:
   Category: N/A
   Pages: 1 - 4
   (No custom fields extracted for this category)

Segment 2:
   Category: Loan application
   Pages: 4 - 4

   Extracted Fields:
      • ApplicationDate: 2025-07-14
      • ApplicantName: John Smith
      • LoanAmountRequested: 25000
      • LoanPurpose: Debt Consolidation
      • CreditScore: None
      • Summary: Loan application for John Smith requesting $25,000 for Debt Consolidation; employed as a Software Engineer at Contoso Technologies with monthly income of $6,500 and 5 years employment length; application signed and dated July 14, 2025.


Document Information:
Total pages in document: 4
Page dimensions: 8.5 x 11 inch


## Saving Classification Results
The classification result is saved to a JSON file for later analysis.

In [12]:
# Save the enhanced analysis result to a file
saved_file_path = save_json_to_file(enhanced_analysis_result, filename_prefix="enhanced_classification_get_result")
# Print the full analysis result as a JSON string
print(json.dumps(enhanced_analysis_result, indent=2)[:1800])


Analysis result saved to: test_output\enhanced_classification_get_result_20261106_083653.json
{
  "id": "65fe2476-8096-4838-9ed3-b3f1a5e61386",
  "status": "Succeeded",
  "result": {
    "analyzerId": "notebooks_sample_enhanced_classifier_1793954201",
    "apiVersion": "2025-11-01",
    "createdAt": "2026-11-06T08:36:44Z",
    "stringEncoding": "codePoint",
    "warnings": [],
    "contents": [
      {
        "path": "input1",
        "markdown": "CONTOSO LTD.\n\n\n# INVOICE\n\nContoso Headquarters\n123 456th St\nNew York, NY, 10001\n\nINVOICE: INV-100\n\nINVOICE DATE: 11/15/2019\n\nDUE DATE: 12/15/2019\n\nCUSTOMER NAME: MICROSOFT CORPORATION\n\nSERVICE PERIOD: 10/14/2019 - 11/14/2019\n\nCUSTOMER ID: CID-12345\n\nMicrosoft Corp\n123 Other St,\nRedmond WA, 98052\n\nBILL TO:\nMicrosoft Finance\n123 Bill St,\nRedmond WA, 98052\n\nSHIP TO:\nMicrosoft Delivery\n123 Ship St,\nRedmond WA, 98052\n\nSERVICE ADDRESS:\nMicrosoft Services\n123 Service St,\nRedmond WA, 98052\n\n\n<table>\n<tr>\n<t

## Clean up the created analyzer
After the demo completes, the analyzer is automatically deleted to prevent resource accumulation.

In [13]:
if RUN_SAMPLE_CLEANUP:
    # Clean up the custom loan analyzer
    print(f"Deleting analyzer '{loan_analyzer_id}'...")
    client.delete_analyzer(analyzer_id=loan_analyzer_id)
    print(f"Analyzer '{loan_analyzer_id}' deleted successfully!")
else:
    print('Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.')



Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.


## Clean up the created classifier
After the demo completes, the classifier is automatically deleted to prevent resource accumulation.

In [14]:
if RUN_SAMPLE_CLEANUP:
    # Clean up the enhanced classifier
    print(f"Deleting classifier '{enhanced_classifier_id}'...")
    client.delete_analyzer(analyzer_id=enhanced_classifier_id)
    print(f"Classifier '{enhanced_classifier_id}' deleted successfully!")
else:
    print('Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.')



Sample analyzer cleanup skipped. IDs are retained for review; set RUN_SAMPLE_CLEANUP=True explicitly to delete only these sample analyzers.
